# 🔴 Solution: DoRA Linear Layer

### Exercise contract
Expose public `nn.Parameter`s `self.A` `(rank, in_features)`, `self.B` `(out_features, rank)`, and `self.m` `(out_features,)`; expose the base direction as `self.W` `(out_features, in_features)`, and keep `self.rank` / `self.alpha`. W may be frozen or trainable. Initialize A randomly and B to zero. These names let tests install known weights; no other module names are required.

For positive rank, form `direction = W + (alpha / rank) * (B @ A)`. Normalize **each output row** along `dim=1`, then multiply by `m[:, None]`. Initialize m from the row norms of W. Forward is bias-free: `x @ effective_weight.T`, mapping `(..., in_features)` to `(..., out_features)`. Support autograd for inputs and m/A/B. The reference also supports rank zero by omitting the update.

In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✅ SOLUTION

class DoRALinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, rank: int, alpha: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.alpha = alpha
        
        # Base weight V (direction)
        self.W = nn.Parameter(torch.empty(out_features, in_features))
        nn.init.kaiming_uniform_(self.W)
        
        # Magnitude vector (initialized from W's norm)
        self.m = nn.Parameter(self.W.norm(dim=1))
        
        # LoRA matrices
        self.A = nn.Parameter(torch.empty(rank, in_features))
        self.B = nn.Parameter(torch.zeros(out_features, rank))
        nn.init.kaiming_uniform_(self.A)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute effective weight: m * (V + BA) / ||V + BA||
        V = self.W
        BA = self.B @ self.A * (self.alpha / self.rank) if self.rank > 0 else 0
        W_eff = V + BA
        W_eff = self.m.view(-1, 1) * W_eff / W_eff.norm(dim=1, keepdim=True)
        return x @ W_eff.T

In [ ]:
# Verify
dora = DoRALinear(in_features=64, out_features=128, rank=8)
x = torch.randn(2, 10, 64)
out = dora(x)
print(f"Output shape: {out.shape}")

In [ ]:
from torch_judge import check
check("dora")